In [1]:
import torch
import torch.nn as nn
from torch.nn import functional as F
from datasets import load_dataset
import matplotlib.pyplot as plt
import math

# 1. Load the dataset (streaming or small slice for fast local training)
dataset = load_dataset("roneneldan/TinyStories", split="train[:2000]")

print(f"Total loaded stories: {len(dataset)}")
for i in range(2):
    print(f"\n--- Story {i+1} (Length: {len(dataset[i]['text'])} chars) ---")
    print(dataset[i]['text'][:300] + "...")

# Combine subset into one text corpus
text = "\n\n".join([item['text'] for item in dataset])
print(f"\nTotal characters in corpus: {len(text):,}")

# 2. Preprocess the text (Character-level tokenizer)
chars = sorted(list(set(text)))
vocab_size = len(chars)
print(f"Vocabulary size: {vocab_size} unique characters")

stoi = {ch: i for i, ch in enumerate(chars)}
itos = {i: ch for i, ch in enumerate(chars)}
encode = lambda s: [stoi[c] for c in s]
decode = lambda l: ''.join([itos[i] for i in l])

# Encode entire dataset and split 90% train / 10% validation
data = torch.tensor(encode(text), dtype=torch.long)
n = int(0.9 * len(data))
train_data = data[:n]
val_data = data[n:]

# 3. Preparing training data (Next-token prediction batches)
batch_size = 32
block_size = 64  # Context window length
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {device}")

def get_batch(split):
    d = train_data if split == 'train' else val_data
    ix = torch.randint(len(d) - block_size, (batch_size,))
    x = torch.stack([d[i:i+block_size] for i in ix])
    y = torch.stack([d[i+1:i+block_size+1] for i in ix])
    return x.to(device), y.to(device)

xb, yb = get_batch('train')
print("Sample input shape:", xb.shape, "| Target shape:", yb.shape)


c:\Users\markg\PUSL3205-Large-Language-Models-and-Generative-AI\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\Users\markg\PUSL3205-Large-Language-Models-and-Generative-AI\.venv\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\markg\.cache\huggingface\hub. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an adm

Total loaded stories: 2000

--- Story 1 (Length: 701 chars) ---
One day, a little girl named Lily found a needle in her room. She knew it was difficult to play with it because it was sharp. Lily wanted to share the needle with her mom, so she could sew a button on her shirt.

Lily went to her mom and said, "Mom, I found this needle. Can you share it with me and ...

--- Story 2 (Length: 705 chars) ---
Once upon a time, there was a little car named Beep. Beep loved to go fast and play in the sun. Beep was a healthy car because he always had good fuel. Good fuel made Beep happy and strong.

One day, Beep was driving in the park when he saw a big tree. The tree had many leaves that were falling. Bee...

Total characters in corpus: 1,704,918
Vocabulary size: 80 unique characters
Using device: cpu
Sample input shape: torch.Size([32, 64]) | Target shape: torch.Size([32, 64])
